Here, I have been testing to see if the issue with the code is formatting or framing (i.e baud rate, parity, start/stop bits, voltage mismatch etc.

The garbled bytes responses from the below cells rule out the possibility of this being a formatting (test by changing what test_cmd is) or framing (second cell cycles different framing paradigms) issue. All thats left is voltage mismatch - which in 99% of cases is the hardware.

In [7]:
import serial
import time

# --- CONFIGURE PORT ---
PORT = "COM4"  # update if needed
BAUD = 9600

# --- CONNECT ---
try:
    ser = serial.Serial(
        port=PORT,
        baudrate=BAUD,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE,
        stopbits=serial.STOPBITS_ONE,
        timeout=1
    )
    print(f"[OK] Connected to {PORT}")
except Exception as e:
    print(f"[ERROR] Could not open port: {e}")
    ser = None

# --- TEST COMMAND ---
if ser and ser.is_open:
    test_cmd = b"\x02@00VER\r"  
    print(f"Sent: {test_cmd}")
    ser.reset_input_buffer()
    ser.write(test_cmd)
    time.sleep(0.2)

    reply = ser.read(64)
    print("Raw bytes:", reply)
    print("Hex:", reply.hex())
    
    ser.close()
else:
    print("Serial port not open.")




[OK] Connected to COM4
Sent: b'\x02@00VER\r'
Raw bytes: b'\xbb'
Hex: bb


In [5]:
import serial, time

PORT = "COM4"
for baud in [9600, 19200]:
    for parity in [serial.PARITY_NONE, serial.PARITY_EVEN]:
        for stop in [serial.STOPBITS_ONE, serial.STOPBITS_TWO]:
            print(f"\nTesting: {baud} baud, {parity} parity, {stop} stop bits")
            try:
                ser = serial.Serial(port=PORT, baudrate=baud, bytesize=8,
                                    parity=parity, stopbits=stop, timeout=1)
                ser.write(b"@00VER\r")
                time.sleep(0.2)
                reply = ser.read(64)
                print("Raw:", reply, "Hex:", reply.hex())
                ser.close()
            except Exception as e:
                print("Error:", e)



Testing: 9600 baud, N parity, 1 stop bits
Raw: b'\xff\xff' Hex: ffff

Testing: 9600 baud, N parity, 2 stop bits
Raw: b'\xff' Hex: ff

Testing: 9600 baud, E parity, 1 stop bits
Raw: b'\xff' Hex: ff

Testing: 9600 baud, E parity, 2 stop bits
Raw: b'\xff\xff' Hex: ffff

Testing: 19200 baud, N parity, 1 stop bits
Raw: b'\xff\xff\xff' Hex: ffffff

Testing: 19200 baud, N parity, 2 stop bits
Raw: b'\xff\xff\xff\xff' Hex: ffffffff

Testing: 19200 baud, E parity, 1 stop bits
Raw: b'\xff\xff\xff' Hex: ffffff

Testing: 19200 baud, E parity, 2 stop bits
Raw: b'\xff\xff\xff\xff' Hex: ffffffff
